# Whisper Turbo - Speech-to-Text

This notebook takes the already processed audio from `dataset/processed/`, transcribes it with Whisper Turbo, and saves `.txt` transcripts in `outputs/whisper/`.

Pipeline:

`dataset/processed/` → `Whisper Turbo` → `outputs/whisper/`


## Step 1 - Import libraries


In [1]:
%pip -q install faster_whisper whisper

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from faster_whisper import WhisperModel


## Step 2 - Set input and output folders

The input is the processed dataset. The raw audio is not used here.


In [3]:
PROCESSED_DIR = Path("dataset/processed")
OUTPUT_DIR = Path("outputs/whisper")

for split in ["train", "test"]:
    (OUTPUT_DIR / split).mkdir(parents=True, exist_ok=True)

print("Input:", PROCESSED_DIR)
print("Output:", OUTPUT_DIR)


Input: dataset/processed
Output: outputs/whisper


## Step 3 - Load Whisper Turbo

This configuration runs Whisper Turbo on the CPU using INT8 computation.


In [4]:
print("Loading Whisper Turbo...")

model = WhisperModel(
    "turbo",
    device="cpu",
    compute_type="int8"
)

print("Whisper Turbo loaded successfully.")


Loading Whisper Turbo...
Whisper Turbo loaded successfully.


## Step 4 - Find processed audio files

The notebook searches both `train` and `test` folders.


In [5]:
AUDIO_EXTENSIONS = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}

audio_files = []

for split in ["train", "test"]:
    split_dir = PROCESSED_DIR / split

    if not split_dir.exists():
        print(f"Folder not found: {split_dir}")
        continue

    for file in split_dir.rglob("*"):
        if file.is_file() and file.suffix.lower() in AUDIO_EXTENSIONS:
            audio_files.append((split, file))

print(f"Found {len(audio_files)} processed audio files.")

for split, file in audio_files:
    print(f"[{split}] {file}")


Found 100 processed audio files.
[train] dataset/processed/train/audio_053.wav
[train] dataset/processed/train/audio_047.wav
[train] dataset/processed/train/audio_046.wav
[train] dataset/processed/train/audio_052.wav
[train] dataset/processed/train/audio_044.wav
[train] dataset/processed/train/audio_050.wav
[train] dataset/processed/train/audio_078.wav
[train] dataset/processed/train/audio_079.wav
[train] dataset/processed/train/audio_051.wav
[train] dataset/processed/train/audio_045.wav
[train] dataset/processed/train/audio_069.wav
[train] dataset/processed/train/audio_041.wav
[train] dataset/processed/train/audio_055.wav
[train] dataset/processed/train/audio_054.wav
[train] dataset/processed/train/audio_040.wav
[train] dataset/processed/train/audio_068.wav
[train] dataset/processed/train/audio_056.wav
[train] dataset/processed/train/audio_042.wav
[train] dataset/processed/train/audio_043.wav
[train] dataset/processed/train/audio_057.wav
[train] dataset/processed/train/audio_030.wav
[

## Step 5 - Transcribe and save TXT files

Each processed audio file is transcribed and saved with the same filename but with a `.txt` extension.


In [6]:
for split, audio_file in audio_files:
    print("=" * 60)
    print(f"Processing: {audio_file.name}")
    print(f"Split: {split}")
    print("=" * 60)

    try:
        segments, info = model.transcribe(
            str(audio_file),
            beam_size=5
        )

        # Complete the transcription
        segments = list(segments)

        relative_path = audio_file.relative_to(PROCESSED_DIR / split)
        output_file = OUTPUT_DIR / split / relative_path.with_suffix(".txt")
        output_file.parent.mkdir(parents=True, exist_ok=True)

        with open(output_file, "w", encoding="utf-8") as f:
            f.write(f"Audio File: {audio_file.name}\n")
            f.write(f"Detected Language: {info.language}\n")
            f.write(f"Language Probability: {info.language_probability:.4f}\n\n")
            f.write("TRANSCRIPT\n")
            f.write("-" * 60 + "\n\n")

            for segment in segments:
                text = segment.text.strip()
                f.write(
                    f"[{segment.start:.2f}s - {segment.end:.2f}s] {text}\n"
                )

        print(f"Transcript saved to: {output_file}")
        print()

    except Exception as e:
        print(f"ERROR processing {audio_file}")
        print(e)
        print()

print("Whisper transcription completed.")


Processing: audio_053.wav
Split: train
Transcript saved to: outputs/whisper/train/audio_053.txt

Processing: audio_047.wav
Split: train
Transcript saved to: outputs/whisper/train/audio_047.txt

Processing: audio_046.wav
Split: train
Transcript saved to: outputs/whisper/train/audio_046.txt

Processing: audio_052.wav
Split: train
Transcript saved to: outputs/whisper/train/audio_052.txt

Processing: audio_044.wav
Split: train
Transcript saved to: outputs/whisper/train/audio_044.txt

Processing: audio_050.wav
Split: train
Transcript saved to: outputs/whisper/train/audio_050.txt

Processing: audio_078.wav
Split: train
Transcript saved to: outputs/whisper/train/audio_078.txt

Processing: audio_079.wav
Split: train
Transcript saved to: outputs/whisper/train/audio_079.txt

Processing: audio_051.wav
Split: train
Transcript saved to: outputs/whisper/train/audio_051.txt

Processing: audio_045.wav
Split: train
Transcript saved to: outputs/whisper/train/audio_045.txt

Processing: audio_069.wav
Spli

## Step 6 - Verify TXT outputs


In [7]:
transcript_files = list(OUTPUT_DIR.rglob("*.txt"))

print(f"Generated {len(transcript_files)} transcript files.")

for file in transcript_files:
    print(file)


Generated 100 transcript files.
outputs/whisper/test/audio_018.txt
outputs/whisper/test/audio_019.txt
outputs/whisper/test/audio_009.txt
outputs/whisper/test/audio_008.txt
outputs/whisper/test/audio_006.txt
outputs/whisper/test/audio_012.txt
outputs/whisper/test/audio_013.txt
outputs/whisper/test/audio_007.txt
outputs/whisper/test/audio_011.txt
outputs/whisper/test/audio_005.txt
outputs/whisper/test/audio_004.txt
outputs/whisper/test/audio_010.txt
outputs/whisper/test/audio_014.txt
outputs/whisper/test/audio_000.txt
outputs/whisper/test/audio_001.txt
outputs/whisper/test/audio_015.txt
outputs/whisper/test/audio_003.txt
outputs/whisper/test/audio_017.txt
outputs/whisper/test/audio_016.txt
outputs/whisper/test/audio_002.txt
outputs/whisper/train/audio_078.txt
outputs/whisper/train/audio_044.txt
outputs/whisper/train/audio_050.txt
outputs/whisper/train/audio_051.txt
outputs/whisper/train/audio_045.txt
outputs/whisper/train/audio_079.txt
outputs/whisper/train/audio_053.txt
outputs/whisper/